In [1]:
import os

In [2]:
%pwd

'c:\\1Kumawat\\Online_learning\\Projects_uploaded_git\\datascience_project_1\\reseach'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\1Kumawat\\Online_learning\\Projects_uploaded_git\\datascience_project_1'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path        #from the config yaml file 
    train_data_path: Path
    test_data_path: Path
    model_name: str    
    alpha: float
    l1_ratio: float
    target_column: str


In [6]:
from src.data_science_project_1.constants import *
from src.data_science_project_1.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self,
                config_file_path= CONFIG_FILE_PATH, # CONFIG_FILE_PATH is a constant that contains the path to the config.yaml file
                params_file_path= PARAMS_FILE_PATH,
                schema_file_path= SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path) # read_yaml is a function that reads a YAML file and returns the data as a dictionary
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_roots]) # artifacts_root is the root directory where all the artifacts will be stored

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet   # available in params.yaml file
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir]) # create_directories is a function that creates directories if they do not exist

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name  # schema.name is the name of the target column in the schema.yaml file
        )
        return model_trainer_config

In [13]:
import pandas as pd
import os
from src.data_science_project_1 import logger
from sklearn.linear_model import ElasticNet
import joblib

In [14]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        
        self.config = config

    def train(self):
        logger.info("Loading training and testing data")
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop(columns=[self.config.target_column], axis=1)  # axis=1 indicates that we are dropping a column, not a row
        test_x = test_data.drop(columns=[self.config.target_column], axis=1)  # drop the target column from the training and testing data
        train_y = train_data[self.config.target_column]
        test_y = test_data[self.config.target_column]
        
        lr = ElasticNet(
            alpha=self.config.alpha,
            l1_ratio=self.config.l1_ratio,
            random_state=42  # random_state is set for reproducibility
        )                                        # ElasticNet is a linear regression model that combines L1 and L2 regularization
        logger.info("Training the model")
        lr.fit(train_x, train_y)         # fit method trains the model on the training data

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))  # joblib is used to save the model to a file
        logger.info(f"Model is saved at {self.config.root_dir}/{self.config.model_name}")


In [15]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    logger.exception(e)
    raise e

[2025-06-23 16:59:43,060: INFO: common: YAML file loaded successfully: config\config.yaml loaded successfully.]
[2025-06-23 16:59:43,063: INFO: common: YAML file loaded successfully: params.yaml loaded successfully.]
[2025-06-23 16:59:43,069: INFO: common: YAML file loaded successfully: schema.yaml loaded successfully.]
[2025-06-23 16:59:43,070: INFO: common: Directory created at: artifacts]
[2025-06-23 16:59:43,072: INFO: common: Directory created at: artifacts/model_trainer]
[2025-06-23 16:59:43,073: INFO: 89249290: Loading training and testing data]
[2025-06-23 16:59:43,094: INFO: 89249290: Training the model]
[2025-06-23 16:59:43,111: INFO: 89249290: Model is saved at artifacts/model_trainer/model.joblib]
